# 07b — Simulation Parameter Spaces

## The One-Sentence Version
Your simulation's parameter space IS a high-dimensional space, and everything we've learned applies.

## What You'll Build Intuition For
- Why simulation parameter spaces are dimensionality problems
- Latin Hypercube Sampling vs grid sampling
- Sensitivity analysis as feature selection
- Surrogate modelling as compression

## Prerequisites
- [07a — Decision Framework](07a_decision_framework.ipynb)
- [05a — Filter Methods](../05_feature_selection/05a_filter_methods.ipynb) — the feature selection connection

## The Story

Suppose you're building a simulation — a casualty evacuation, a hospital queue, a manufacturing line. The simulation has parameters: arrival rates, service times, capacities, distances, probabilities. Each parameter is a dimension.

With 15 parameters, even trying 5 values each gives 5^15 ≈ 30 billion scenarios. You can't run them all. You need to reduce dimensions.

This is the same problem we've been solving all along. Sensitivity analysis is feature selection. A surrogate model is a compressed representation. Latin Hypercube Sampling is efficient exploration of high-dimensional space. The tools are the same — only the language changes.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import qmc
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel

from utils.plotting import apply_style, COLOURS, PALETTE

apply_style()
rng = np.random.default_rng(42)

## Simulation as Dimensionality Problem

A simulation with 15 parameters lives in a 15-dimensional space. Each run is a point in that space. The output (e.g., average wait time) is a function $f: \mathbb{R}^{15} \to \mathbb{R}$.

In [ ]:
# Define a simple simulation: casualty evacuation scenario
param_names = [
    "arrival_rate", "triage_time", "treatment_time", "transport_time",
    "bed_capacity", "staff_count", "equipment_level", "distance_km",
    "severity_mix", "weather_factor", "comms_reliability", "road_quality",
    "backup_capacity", "shift_length", "fatigue_factor"
]

n_params = len(param_names)

# A toy simulation function: only 3-4 parameters actually matter
def simulate(params):
    """Toy simulation: output depends mainly on arrival_rate, treatment_time,
    bed_capacity, and a bit on staff_count. Other parameters are noise."""
    arrival = params[:, 0]
    treatment = params[:, 2]
    beds = params[:, 4]
    staff = params[:, 5]

    # Core dynamics: wait time increases when demand exceeds capacity
    utilisation = (arrival * treatment) / (beds * staff + 1)
    wait_time = 10 * utilisation ** 2 + rng.normal(0, 1, len(arrival))
    return wait_time.clip(0)

# Show the curse of dimensionality
levels = 5
total_grid = levels ** n_params
print(f"Parameters: {n_params}")
print(f"Grid search at {levels} levels each: {levels}^{n_params} = {total_grid:,.0f} scenarios")
print(f"At 1 second per run: {total_grid / 3600 / 24 / 365:.0f} years")
print(f"\nWe MUST reduce dimensions to explore this space.")

## Latin Hypercube Sampling

Instead of a grid, sample efficiently. Latin Hypercube Sampling (LHS) divides each dimension into equal intervals and ensures every interval is sampled exactly once — giving much better coverage than random or grid sampling with the same budget.

In [ ]:
# Compare grid vs random vs LHS in 2D
n_samples = 25

# Grid
grid_1d = np.linspace(0.1, 0.9, int(np.sqrt(n_samples)))
grid_x, grid_y = np.meshgrid(grid_1d, grid_1d)
grid_points = np.column_stack([grid_x.ravel(), grid_y.ravel()])

# Random
random_points = rng.uniform(0, 1, (n_samples, 2))

# LHS
sampler = qmc.LatinHypercube(d=2, seed=42)
lhs_points = sampler.random(n=n_samples)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 5))

ax1.scatter(grid_points[:, 0], grid_points[:, 1], s=40, color=COLOURS["noise"],
            edgecolors="white", linewidth=0.5)
ax1.set_title(f"Grid ({len(grid_points)} points)\nGood coverage, wastes budget")
ax1.set_xlim(0, 1)
ax1.set_ylim(0, 1)
ax1.set_aspect("equal")

ax2.scatter(random_points[:, 0], random_points[:, 1], s=40, color=COLOURS["accent"],
            edgecolors="white", linewidth=0.5)
ax2.set_title(f"Random ({n_samples} points)\nGaps and clumps")
ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)
ax2.set_aspect("equal")

ax3.scatter(lhs_points[:, 0], lhs_points[:, 1], s=40, color=COLOURS["signal"],
            edgecolors="white", linewidth=0.5)
ax3.set_title(f"LHS ({n_samples} points)\nBest coverage per sample")
ax3.set_xlim(0, 1)
ax3.set_ylim(0, 1)
ax3.set_aspect("equal")

fig.suptitle("Sampling strategies: LHS gives the best bang for your budget",
             fontweight="bold")
plt.tight_layout()
plt.savefig("visuals/07b_sampling.png", dpi=150, bbox_inches="tight")
plt.show()

print("LHS ensures each row and column of the grid is sampled exactly once.")
print("This eliminates gaps and clusters that plague random sampling.")

## Sensitivity Analysis as Feature Selection

Most simulation parameters barely matter. Sensitivity analysis identifies the 3-5 that actually drive the output. This is *exactly* feature selection — applied to simulation.

In [ ]:
# Generate LHS samples in 15D and run the simulation
n_runs = 500
sampler = qmc.LatinHypercube(d=n_params, seed=42)
X_lhs = sampler.random(n=n_runs)

# Scale to realistic parameter ranges
param_ranges = {
    "arrival_rate": (1, 20), "triage_time": (2, 15), "treatment_time": (10, 120),
    "transport_time": (5, 60), "bed_capacity": (5, 50), "staff_count": (2, 20),
    "equipment_level": (1, 10), "distance_km": (1, 100), "severity_mix": (0.1, 0.9),
    "weather_factor": (0.5, 1.5), "comms_reliability": (0.5, 1.0), "road_quality": (0.3, 1.0),
    "backup_capacity": (0, 20), "shift_length": (6, 12), "fatigue_factor": (0.8, 1.2)
}

X_scaled = np.zeros_like(X_lhs)
for j, name in enumerate(param_names):
    lo, hi = param_ranges[name]
    X_scaled[:, j] = lo + X_lhs[:, j] * (hi - lo)

# Run simulation
y_sim = simulate(X_scaled)

# Simple sensitivity: correlation between each parameter and output
correlations = np.array([np.corrcoef(X_scaled[:, j], y_sim)[0, 1]
                          for j in range(n_params)])

# Lasso for importance
scaler = StandardScaler()
X_std = scaler.fit_transform(X_scaled)
lasso = Lasso(alpha=0.1, max_iter=10000)
lasso.fit(X_std, y_sim)

# Plot both
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

order_corr = np.argsort(np.abs(correlations))[::-1]
important_corr = np.abs(correlations[order_corr]) > 0.1
bar_colours_corr = [COLOURS["signal"] if imp else COLOURS["noise"]
                     for imp in important_corr]
ax1.barh([param_names[i] for i in order_corr], np.abs(correlations[order_corr]),
         color=bar_colours_corr, alpha=0.8, edgecolor="white")
ax1.set_xlabel("|Correlation with output|")
ax1.set_title("Correlation-based sensitivity")
ax1.invert_yaxis()

order_lasso = np.argsort(np.abs(lasso.coef_))[::-1]
important_lasso = np.abs(lasso.coef_[order_lasso]) > 0.01
bar_colours_lasso = [COLOURS["signal"] if imp else COLOURS["noise"]
                      for imp in important_lasso]
ax2.barh([param_names[i] for i in order_lasso], np.abs(lasso.coef_[order_lasso]),
         color=bar_colours_lasso, alpha=0.8, edgecolor="white")
ax2.set_xlabel("|Lasso coefficient|")
ax2.set_title("Lasso-based sensitivity")
ax2.invert_yaxis()

fig.suptitle("Sensitivity analysis: most parameters barely matter", fontweight="bold")
plt.tight_layout()
plt.savefig("visuals/07b_sensitivity.png", dpi=150, bbox_inches="tight")
plt.show()

print("Only 3-4 parameters significantly influence the output.")
print("The rest can be fixed at default values without losing insight.")
print("\nThis reduces our effective search space from 15D to ~4D — manageable.")

## Surrogate Modelling as Compression

A surrogate model is a fast approximation of the simulation. It's trained on a small set of simulation runs and then used to *predict* the output for new parameter combinations without running the expensive simulation.

The surrogate IS a compressed representation of the simulation's input-output relationship.

In [ ]:
# Train a Gaussian Process surrogate on the important parameters only
important_params = [0, 2, 4, 5]  # arrival_rate, treatment_time, bed_capacity, staff_count
X_important = X_std[:, important_params]

# Split: 80% train, 20% test
n_train = int(0.8 * n_runs)
X_train, X_test = X_important[:n_train], X_important[n_train:]
y_train, y_test = y_sim[:n_train], y_sim[n_train:]

# Fit GP surrogate
kernel = ConstantKernel(1.0) * RBF(length_scale=1.0)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1.0, random_state=42)
gp.fit(X_train, y_train)

# Predict
y_pred, y_std = gp.predict(X_test, return_std=True)

fig, ax = plt.subplots(figsize=(8, 8))
ax.errorbar(y_test, y_pred, yerr=2 * y_std, fmt='o', color=COLOURS["signal"],
            alpha=0.5, markersize=5, ecolor=COLOURS["accent"], elinewidth=0.5)
lims = [min(y_test.min(), y_pred.min()) - 1, max(y_test.max(), y_pred.max()) + 1]
ax.plot(lims, lims, '--', color=COLOURS["noise"], linewidth=1.5, label="Perfect prediction")
ax.set_xlabel("Actual simulation output")
ax.set_ylabel("Surrogate prediction")
ax.set_title("Surrogate model: fast approximation of the simulation")
ax.legend()
ax.set_aspect("equal")
plt.tight_layout()
plt.savefig("visuals/07b_surrogate.png", dpi=150, bbox_inches="tight")
plt.show()

r2 = 1 - np.sum((y_test - y_pred)**2) / np.sum((y_test - y_test.mean())**2)
print(f"Surrogate R² on test set: {r2:.3f}")
print(f"Using only {len(important_params)} of {n_params} parameters.")
print(f"\nThe surrogate runs in microseconds vs seconds for the real simulation.")
print(f"We can explore millions of scenarios using the surrogate.")

## Design Space Exploration

With sensitivity analysis telling us which parameters matter and a surrogate model for fast prediction, we can now efficiently explore the design space.

In [ ]:
# Explore the reduced design space using the surrogate
# Vary the 2 most important parameters, fix the rest at median values
n_grid = 50
p1_range = np.linspace(X_important[:, 0].min(), X_important[:, 0].max(), n_grid)
p2_range = np.linspace(X_important[:, 1].min(), X_important[:, 1].max(), n_grid)

P1, P2 = np.meshgrid(p1_range, p2_range)
grid_points = np.column_stack([
    P1.ravel(), P2.ravel(),
    np.full(n_grid**2, np.median(X_important[:, 2])),
    np.full(n_grid**2, np.median(X_important[:, 3])),
])

y_surface = gp.predict(grid_points).reshape(n_grid, n_grid)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.contourf(P1, P2, y_surface, levels=20, cmap="RdYlBu_r")
plt.colorbar(im, ax=ax, label="Predicted Wait Time")
ax.set_xlabel("Arrival Rate (standardised)")
ax.set_ylabel("Treatment Time (standardised)")
ax.set_title("Design space exploration: surrogate surface\n(bed capacity & staff fixed at median)")
plt.tight_layout()
plt.savefig("visuals/07b_design_space.png", dpi=150, bbox_inches="tight")
plt.show()

print("The surface reveals the operating regions:")
print("  Blue: low wait times (system is adequately resourced)")
print("  Red: high wait times (system is overwhelmed)")
print("  The boundary between them is where interesting decisions happen.")

> **Key insight:** The workflow — LHS sampling → sensitivity analysis → surrogate model → design space exploration — is dimensionality reduction applied to simulation. We reduced a 15D problem to a 4D one, built a compressed representation (surrogate), and explored the low-dimensional subspace where the action is.

<details>
<summary><b>The Maths Behind This</b></summary>

**Latin Hypercube Sampling:** Divides each dimension into $n$ equal-probability intervals and samples one point per interval per dimension, creating a permutation matrix structure. This guarantees marginal stratification: each 1D projection has exactly one point per interval.

**Sobol indices:** Decompose output variance into contributions from each parameter and their interactions:

$$\text{Var}(Y) = \sum_i V_i + \sum_{i<j} V_{ij} + \ldots$$

First-order Sobol index: $S_i = V_i / \text{Var}(Y)$ measures the fraction of output variance caused by parameter $i$ alone.

**Gaussian Process surrogate:** Models the simulation output as a sample from a Gaussian process:

$$f(\mathbf{x}) \sim \mathcal{GP}(m(\mathbf{x}), k(\mathbf{x}, \mathbf{x}'))$$

The kernel $k$ defines similarity between parameter configurations. Prediction at a new point comes with uncertainty estimates — the GP knows what it doesn't know.

</details>

## Where This Matters

**Defence simulation (FAER-M):** A casualty evacuation scenario might have 20+ parameters: evacuation time, triage accuracy, treatment capacity, transport distance, weather, communications reliability. Sensitivity analysis reveals that only 3-5 of these drive the outcome. Decision-makers focus resources on controlling those parameters.

**Manufacturing optimisation:** A factory simulation with 50+ machine settings. LHS + sensitivity analysis identifies the 5 settings that affect throughput. Engineers tune those 5 instead of running blind experiments across all 50.

## Feynman Check

1. **Why is grid search impossible for a simulation with 15 parameters?**

2. **How is sensitivity analysis a form of dimensionality reduction?**

3. **What's the connection between a surrogate model and a compressed representation?**

---

We've applied dimensionality thinking to simulation. In **07c — Compression as Philosophy**, we'll zoom all the way out and see how compression is a fundamental principle of intelligence itself.